# HWP (한글) 문서 로더

한글(HWP)은 한글과컴퓨터에서 개발한 워드프로세서로, 한국에서 가장 널리 사용되는 문서 형식입니다.

공공기관, 기업, 학교 등에서 광범위하게 사용되기 때문에 한국 개발자라면 HWP 파일을 처리해야 하는 경우가 많습니다.


# 02. HWP(한글) 문서 로더

## 학습 목표
- HWP 파일의 내부 구조(OLE 복합 문서 형식)를 이해해요
- `UnstructuredFileLoader`와 커스텀 로더 두 가지 방법으로 HWP를 로딩해요
- 상황에 맞는 HWP 처리 전략을 선택할 수 있어요

## 사전 지식
- 00-Document-Loader 노트북 완료
- `pip install unstructured olefile` 설치

---

> **RAG 파이프라인 위치**: Load 단계 — 이번에는 한국에서 널리 쓰이는 **HWP 형식**을 처리해요.

```mermaid
flowchart LR
    A[📄 Load<br/>HWP 문서]:::current --> B[✂️ Chunk]:::process
    B --> C[🔢 Embed]:::process
    C --> D[🗄️ Store]:::storage
    D --> E[🔍 Retrieve]:::process
    E --> F[💬 Generate]:::output

    classDef current fill:#d4edda,stroke:#28a745,color:#155724
    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef output fill:#fff3cd,stroke:#ffc107,color:#856404
    classDef storage fill:#e2d5f1,stroke:#6f42c1,color:#3d1f6e
```


## 학습 목표

- HWP 파일의 특징과 구조 이해
- HWP 파일을 Document 객체로 변환하는 방법 학습
- HWP 로더 직접 구현하기


## HWP 파일 특징

### HWP 파일이란?

- **파일 확장자**: `.hwp`
- **개발사**: 한글과컴퓨터
- **용도**: 워드프로세싱 (문서 작성, 편집)
- **특징**: 한글 처리에 최적화, 복잡한 레이아웃 지원

### HWP 파일 구조

HWP 파일은 내부적으로 **복합 문서(Compound File Binary Format)** 구조를 사용합니다.

- OLE(Object Linking and Embedding) 형식
- 여러 스트림(stream)으로 구성
- 텍스트, 이미지, 서식 등이 분리되어 저장


## HWP 로더 직접 구현하기

LangChain에는 공식 HWP 로더가 아직 포함되어 있지 않으므로, 직접 구현해야 합니다.

HWP 파일을 파싱하는 방법은 여러 가지가 있습니다:

### 방법 1: Unstructured 라이브러리 사용
- 가장 간단하고 권장되는 방법
- 다양한 문서 형식을 통합 지원

### 방법 2: olefile + 직접 파싱
- HWP 파일 구조에 직접 접근
- 세밀한 제어 가능하지만 복잡함

### 방법 3: hwp5 라이브러리
- HWP 5.0 형식 전용
- 구조화된 접근 가능

이 노트북에서는 **Unstructured 라이브러리**를 사용하는 방법을 다룹니다.


In [ ]:
# 환경 설정
from dotenv import load_dotenv

load_dotenv()

# HWP 파일 경로
FILE_PATH = "./data/디지털 정부혁신 추진계획.hwp"


## 방법 1: Unstructured 사용 (권장)

`Unstructured` 라이브러리는 다양한 문서 형식을 자동으로 감지하고 파싱합니다.

**필요한 패키지**:
```bash
pip install unstructured
pip install python-magic-bin  # Windows용 (선택)
```


In [ ]:
from langchain_community.document_loaders import UnstructuredFileLoader
from unstructured.partition.auto import UnsupportedFileFormatError

# ============================================================
# TODO: UnstructuredFileLoader로 HWP 파일을 로딩해보세요
# 힌트: UnstructuredFileLoader(FILE_PATH)로 초기화 후 load()를 호출합니다
# 예상 결과: 성공 시 Document 내용 출력, 실패 시 안내 메시지 출력
# ============================================================

# 1단계: UnstructuredFileLoader 생성
# - UnstructuredFileLoader: 다양한 문서 형식을 자동 감지하는 범용 로더

# 2단계: 문서 로드


In [ ]:
# 메타데이터 확인


## 방법 2: 커스텀 HWP 로더 클래스 구현

Unstructured가 설치되지 않았거나 더 세밀한 제어가 필요한 경우, 직접 HWP 로더를 구현할 수 있습니다.

**필요한 패키지**:
```bash
pip install olefile zlib
```


In [ ]:
import os
import zlib
import struct
from typing import List
from langchain_core.documents import Document
from langchain_core.document_loaders import BaseLoader

# ---------------------------------------------------
# HWP 파일 커스텀 로더 클래스 정의
# ---------------------------------------------------

class CustomHWPLoader(BaseLoader):
    """
    HWP 파일을 로드하는 커스텀 로더
    
    HWP 파일은 OLE 복합 문서 형식으로 저장되어 있습니다.
    이 로더는 텍스트 스트림을 추출하여 Document 객체로 변환합니다.
    """
    
    def __init__(self, file_path: str):
        """
        Args:
            file_path: HWP 파일 경로
        """
        self.file_path = file_path
        
    def load(self) -> List[Document]:
        """HWP 파일을 로드하여 Document 리스트 반환"""
        try:
            import olefile
        except ImportError:
            raise ImportError(
                "olefile 패키지가 필요합니다. "
                "`pip install olefile`로 설치하세요."
            )
        
        # 1단계: HWP 파일 열기 (OLE 복합 문서 형식 검증)
        if not olefile.isOleFile(self.file_path):
            raise ValueError(f"{self.file_path}는 유효한 HWP 파일이 아닙니다.")
        
        ole = olefile.OleFileIO(self.file_path)
        
        # 2단계: 텍스트 추출
        text_content = self._extract_text(ole)
        
        ole.close()
        
        # 3단계: Document 객체 생성
        metadata = {
            "source": self.file_path,
            "file_name": os.path.basename(self.file_path),
        }
        
        return [Document(page_content=text_content, metadata=metadata)]
    
    def _extract_text(self, ole) -> str:
        """HWP 파일에서 텍스트 추출"""
        text_parts = []
        
        # HWP 파일의 섹션 탐색
        for stream_name in ole.listdir():
            # BodyText 스트림에서 텍스트 추출
            if stream_name[0].startswith("BodyText"):
                try:
                    stream = ole.openstream(stream_name)
                    data = stream.read()
                    
                    # 압축 해제 (HWP 5.0+)
                    try:
                        decompressed = zlib.decompress(data, -15)
                        # 텍스트 디코딩 시도
                        text = self._decode_text(decompressed)
                        if text:
                            text_parts.append(text)
                    except:
                        # 압축되지 않은 데이터 처리
                        text = self._decode_text(data)
                        if text:
                            text_parts.append(text)
                except:
                    continue
        
        return "\n\n".join(text_parts) if text_parts else "텍스트 추출 실패"
    
    def _decode_text(self, data: bytes) -> str:
        """바이트 데이터를 텍스트로 디코딩"""
        # 여러 인코딩 순서대로 시도
        # - utf-16-le: HWP 5.0 기본 인코딩
        # - cp949: Windows 한글 코드 페이지
        # - euc-kr: 구형 한글 인코딩
        encodings = ['utf-16-le', 'cp949', 'euc-kr', 'utf-8']
        
        for encoding in encodings:
            try:
                text = data.decode(encoding, errors='ignore')
                text = ''.join(char for char in text if char.isprintable() or char.isspace())
                if len(text.strip()) > 10:
                    return text.strip()
            except:
                continue
        
        return ""


print("✅ CustomHWPLoader 클래스 정의 완료")

In [ ]:
# 커스텀 HWP 로더 사용


## HWP 파일 처리 시 주의사항

### 1. 버전 호환성

HWP 파일은 여러 버전이 있습니다:
- **HWP 3.0 이하**: 구형 형식, 파싱이 복잡
- **HWP 5.0+**: OLE 형식, 상대적으로 파싱 용이
- **HWPX**: XML 기반 형식, 압축된 XML 파일

### 2. 인코딩 문제

HWP 파일은 한글 처리를 위해 다양한 인코딩을 사용:
- CP949 (Windows 한글 코드 페이지)
- EUC-KR
- UTF-16LE

### 3. 복잡한 레이아웃

HWP는 복잡한 레이아웃을 지원하므로:
- 순수 텍스트만 추출하면 레이아웃 정보 손실
- 테이블, 이미지, 도형 등은 별도 처리 필요
- 페이지 구조가 명확하지 않을 수 있음


## 핵심 정리 및 마무리

### HWP 처리 방법 요약

| 방법 | 권장 상황 | 설치 |
|------|----------|------|
| `UnstructuredFileLoader` | 일반적인 경우 (권장) | `pip install unstructured` |
| 커스텀 로더 (`olefile`) | 세밀한 제어가 필요할 때 | `pip install olefile` |
| HWP → PDF 변환 후 처리 | 가장 확실한 방법 | 별도 변환 도구 필요 |

### 주의 사항
> HWP 파일은 **인코딩 문제**가 자주 발생해요. `cp949`, `euc-kr`, `utf-16-le` 순서로 시도해 보세요. 커스텀 로더에서 `errors='ignore'`를 사용하면 일부 문자가 손실될 수 있으니 결과를 꼭 검증해야 해요.

---

## 다음 예고

다음 노트북에서는 Office 문서 형식을 이어서 다뤄볼게요.

- **03-CSV-Loader**: 표 형식 데이터 로딩
- **04-Excel-Loader**: Excel 파일 로딩
- **05-Word-Loader**: Word 문서 로딩
- **06-PowerPoint-Loader**: 프레젠테이션 파일 로딩
